- **검색 키워드 입력**
- 네이버 뉴스 URL에 키워드 붙여서 요청
- HTML을 BeautifulSoup으로 파싱
- 뉴스 리스트 영역 선택
- 각 뉴스마다
    - 제목, 링크, 이미지, 추출
- 이미지가 있으면 `images/` 폴더에 저장
- NewsEntry 객체로 저장
- 리스트 출력

In [2]:
import requests   #HTML 요청
from bs4 import BeautifulSoup #HTML 파싱 도구
from urllib.request import urlretrieve #이미지 다운로드용
from datetime import datetime #파일명 시간 생성용

In [3]:
#스크랩할 뉴스 정보를 담을 NewsEntry Class
class NewsEntry:
    def __init__(self,title,href, img_path):
        self.title = title        #뉴스 제목
        self.href = href          #뉴스 링크
        self.img_path = img_path  #이미지 경로

    def __repr__(self):
        return f"NewsEntry<title={self.title}, href={self.href}>"    



In [21]:
keyword = input('뉴스 검색 키워드 입력:')

url = f"https://search.naver.com/search.naver?sm=tab_hty.top&where=news&ssc=tab.news.all&query={keyword}"

response = requests.get(url) # get 요청 ---> HTML 응답 받음

response   #보려면 파싱 필요

<Response [200]>

In [22]:
# BeautifulSoup로 html 파싱

html = response.text
bs = BeautifulSoup(html, 'html.parser')
print(bs.prettify())

<!DOCTYPE html>
<html lang="ko">
 <head>
  <meta charset="utf-8"/>
  <meta content="strict-origin-when-cross-origin" name="referrer"/>
  <meta content="telephone=no,address=no,email=no" name="format-detection"/>
  <meta content="태풍 : 네이버 뉴스검색" property="og:title"/>
  <meta content="https://ssl.pstatic.net/sstatic/search/common/og_v3.png" property="og:image"/>
  <meta content="'태풍'의 네이버 뉴스검색 결과입니다." property="og:description"/>
  <meta content="'태풍'의 네이버 뉴스검색 결과입니다." lang="ko" name="description"/>
  <title>
   태풍 : 네이버 뉴스검색
  </title>
  <link href="https://ssl.pstatic.net/sstatic/search/favicon/favicon_32x32_240820.ico" rel="shortcut icon"/>
  <link href="https://ssl.pstatic.net/sstatic/search/opensearch-description.https.xml" rel="search" title="Naver" type="application/opensearchdescription+xml"/>
  <link href="https://ssl.pstatic.net/sstatic/search/pc/css/search1_260625.css?o=search" rel="stylesheet" type="text/css"/>
  <link href="https://ssl.pstatic.net/sstatic/search/pc/css/search2

In [23]:
#뉴스 목록 전체 리스트로 구성
news_contents = bs.select('.fds-news-item-list-tab > div')

news_list = []

news_contents

[<div class="sds-comps-vertical-layout sds-comps-full-layout U7hwzP3DLKcLg5Q7"><div class="sds-comps-horizontal-layout sds-comps-full-layout sds-comps-profile type-basic size-lg title-color-g10 uMsmSRLp9R0NRUKG" data-sds-comp="Profile"><div class="sds-comps-horizontal-layout sds-comps-inline-layout sds-comps-profile-source"><div class="sds-comps-horizontal-layout sds-comps-inline-layout sds-comps-profile-source-thumb"><a class="fender-ui_228e3bd1" data-heatmap-target=".prof" href="https://media.naver.com/press/022" nocr="1" target="_blank"><div class="sds-comps-horizontal-layout sds-comps-inline-layout sds-comps-profile-thumbnail type-basic size-lg" data-sds-comp="ProfileThumbnail"><div class="sds-comps-base-layout sds-comps-inline-layout sds-comps-image sds-comps-image-circle fit-cover is-loading" data-dimmed="3%" data-sds-comp="RectangleImage" style="width:24px;height:24px;aspect-ratio:1/1"><img alt="세계일보의 프로필 이미지" height="24" loading="lazy" src="https://search.pstatic.net/common/?sr

In [28]:
for idx, news_content in enumerate(news_contents):
    
    #제목 태그 선택
    title_tag = news_content.select_one('span.sds-comps-text-type-headline1')

    #링크 태그 선택
    href_tag = title_tag.find_parent('a')

    img_tag = news_content.find_all('img')[1]

    title = title_tag.next
    href = href_tag['href']
    img_path = ''
    
    #실제 이미지가 존재할 시
    if img_tag.has_attr('src'):
        img_path = img_tag['src']

        img_dir = '../images'
        file_name = datetime.now().strftime('%y%m%d_%H%M%S_') + str(idx + 1) + '.jpg'

        urlretrieve(img_path, f'{img_dir}/{file_name}')  #이미지 저장

    #NewsEntry 객체 생성 후 리스트에 저장
    news_entry = NewsEntry(title, href, img_path)
    news_list.append(news_entry)

for news in news_list:
    print(news)

NewsEntry<title=여름철 , href=https://www.segye.com/newsView/20260630519412?OutUrl=naver>
NewsEntry<title=「꽁냥꽁냥 그림과학」, href=https://www.kidshankook.kr/news/articleView.html?idxno=17550>
NewsEntry<title=[2026년 , href=http://www.economytalk.kr/news/articleView.html?idxno=422363>
NewsEntry<title=“, href=https://www.segye.com/newsView/20260630516931?OutUrl=naver>
NewsEntry<title=전력거래소, 여름철 전력수급 비상체제 돌입… "폭염·, href=https://www.energydaily.co.kr/news/articleView.html?idxno=200951>
NewsEntry<title=일본, 6월 첫 '쌍 , href=http://mbn.mk.co.kr/pages/news/newsView.php?category=mbn00008&news_seq_no=5202360>
NewsEntry<title=포항공단 , href=https://www.wikitree.co.kr/articles/1143695>
NewsEntry<title=日 기상청 "2~3일 내 큰 지진 가능성"…, href=https://www.newsis.com/view/NISX20260627_0003685902>
NewsEntry<title=장마·, href=https://www.newspim.com/news/view/20260629000401>
NewsEntry<title=대전 중구, 석교동 '주민대피지원단' 훈련... , href=http://www.newsworker.co.kr/news/articleView.html?idxno=433797>
NewsEntry<title=여름철 , href=https://www.seg